# 04 · LightGBM Coarse View

**Competition:** Kaggle Playground Series S6E9 — Predicting Electric Vehicle Purchases  
**Metric:** ROC-AUC

This notebook develops a deliberately coarser LightGBM feature view to complement the high-resolution `lgb_exact` model.

Rather than maximizing feature resolution, this model emphasizes alternative statistical representations and local target structure.

The model combines:

1. raw numeric features with a coarser income representation;
2. income ladders at divisors `10 / 50 / 500 / 5000`;
3. fold-local frequency features;
4. three cross-fitted Target Encodings (`auto`, `20`, `200`);
5. target statistics from the external original EV dataset after exact-overlap removal;
6. original-data support, novelty, and target-rate features;
7. competition-vs-original log-odds lift;
8. cross-fitted local target-rate windows over income and commute;
9. 5-fold OOF validation;
10. two LightGBM seeds.

This component is intentionally different from `lgb_exact`: its value is evaluated by marginal ensemble contribution rather than standalone AUC alone.

## 1. Imports and configuration

Expected files:

- `train.csv`
- `test.csv`
- `sample_submission.csv`
- `EV_Adoption_and_Range_Anxiety_Dataset.csv`

The original EV dataset supplies external supervised statistics. Before those statistics are constructed, any row whose **full feature vector** exactly matches either competition train or competition test is removed.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.special import logit
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path(".")
ORIGINAL_PATH = Path("EV_Adoption_and_Range_Anxiety_Dataset.csv")
OUTPUT_DIR = Path("outputs/04_lgb_coarse")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment setup
# -----------------------------
TARGET = "Will_Buy_EV"
ID_COL = "id"
INCOME = "Annual_Income_USD"
COMMUTE = "Daily_Commute_km"

OUTER_FOLDS = 5
INNER_FOLDS = 5
FOLD_SEED = 42
MODEL_SEEDS = [42, 152]
THREADS = 8

CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

NUMS = [
    "Age",
    INCOME,
    COMMUTE,
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

RAW = NUMS + CATS

## 2. Load competition and original data

The target is mapped from `No/Yes` to `0/1`.

The overlap filter is feature-only: it does not use competition labels. Its purpose is to prevent an external labeled row from being an exact copy of a competition row used for validation or test prediction.

In [ ]:
def map_target(series):
    mapped = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({
            "yes": 1,
            "no": 0,
            "1": 1,
            "0": 0,
            "1.0": 1,
            "0.0": 0,
        })
    )

    if mapped.isna().any():
        raise ValueError("Target contains values outside Yes/No or 0/1.")

    return mapped.astype("int8")


def canonical_features(frame):
    out = frame[RAW].copy()

    for col in CATS:
        out[col] = (
            out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

    for col in NUMS:
        out[col] = pd.to_numeric(
            out[col],
            errors="raise",
        ).astype("float64")

    return out


train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)
original = pd.read_csv(ORIGINAL_PATH)

y = map_target(train[TARGET]).to_numpy()

competition_full = canonical_features(
    pd.concat(
        [train[RAW], test[RAW]],
        ignore_index=True,
    )
)
original_full = canonical_features(original)

occupied_hash = pd.util.hash_pandas_object(
    competition_full,
    index=False,
)
original_hash = pd.util.hash_pandas_object(
    original_full,
    index=False,
)

overlap_mask = original_hash.isin(
    occupied_hash
)
overlap_removed = int(
    overlap_mask.sum()
)

original = (
    original.loc[~overlap_mask]
    .reset_index(drop=True)
)

original_y = map_target(
    original[TARGET]
).to_numpy()

all_comp = canonical_features(
    pd.concat(
        [train[RAW], test[RAW]],
        ignore_index=True,
    )
)
original_comp = canonical_features(
    original
)

n_train = len(train)
n_total = len(all_comp)
test_idx = np.arange(
    n_train,
    n_total,
)

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Positive rate:", y.mean())
print("Original rows after overlap removal:", len(original))
print("Original overlaps removed:", overlap_removed)

## 3. Static coarse feature view

This branch intentionally differs from `lgb_exact`.

### 3.1 Raw numeric features

All raw numeric variables except `Number_of_Cars_Owned` are retained:

- `Age`
- `Annual_Income_USD`
- `Daily_Commute_km`
- home charging-station count
- work charging-station count
- environmental-concern level

The categorical columns are **not** passed directly to LightGBM in this view. They enter through frequency / Target-Encoding statistics and through the group definition used by the window features.

### 3.2 Coarse income ladder

Instead of fine digit decomposition, this view uses

$$
\left\lfloor
\frac{\mathrm{Income}}{d}
\right\rfloor,
\qquad
d \in \{10,50,500,5000\}.
$$

This produces intentionally coarser partitions than the exact view.

### 3.3 Additional numeric structure

The view also contains:

- `commute_integer`
- `homecharge`
- `is_30k_spike`
- `is_income_cliff`
- `is_low_income_interval`
- `is_env_one`

The selected static numeric matrix contains **16 features**.

In [ ]:
def remove_constant_and_duplicate_columns(frame):
    kept = []
    seen = {}

    for name in frame.columns:
        values = frame[name].to_numpy()

        if frame[name].nunique(
            dropna=False
        ) <= 1:
            continue

        digest = hashlib.blake2b(
            np.ascontiguousarray(
                values
            ).tobytes(),
            digest_size=12,
        ).digest()

        duplicate = any(
            np.array_equal(
                values,
                frame[old].to_numpy(),
                equal_nan=True,
            )
            for old in seen.get(
                digest,
                [],
            )
        )

        if duplicate:
            continue

        seen.setdefault(
            digest,
            [],
        ).append(name)

        kept.append(name)

    return frame[kept]


numeric_features = {}
raw_keys = {}
original_keys = {}


def add_key(
    name,
    values,
    original_values=None,
):
    raw_keys[name] = np.asarray(
        values
    )

    if original_values is not None:
        original_keys[name] = np.asarray(
            original_values
        )


def original_column(col):
    return original_comp[
        col
    ].to_numpy()


# -----------------------------
# Raw features / keys
# -----------------------------
for col in RAW:
    # The coarse view intentionally excludes Number_of_Cars_Owned.
    if col == "Number_of_Cars_Owned":
        continue

    competition_values = all_comp[
        col
    ].to_numpy()

    if col in NUMS:
        numeric_features[col] = (
            competition_values
            .astype("float32")
        )

    # Annual income itself is deliberately NOT a TE/frequency key
    # in the coarse branch.
    if col != INCOME:
        add_key(
            col,
            competition_values,
            original_column(col),
        )


income = all_comp[
    INCOME
].to_numpy()

commute = all_comp[
    COMMUTE
].to_numpy()

# -----------------------------
# Coarse income ladder
# -----------------------------
for divisor in [
    10,
    50,
    500,
    5000,
]:
    name = (
        f"income_floor_{divisor}"
    )

    values = np.floor(
        income / divisor
    ).astype("float32")

    original_values = np.floor(
        original_comp[
            INCOME
        ].to_numpy()
        / divisor
    )

    numeric_features[name] = values

    add_key(
        name,
        values,
        original_values,
    )

# -----------------------------
# Integer commute
# -----------------------------
commute_integer = np.floor(
    commute
).astype("float32")

numeric_features[
    "commute_integer"
] = commute_integer

add_key(
    "commute_integer",
    commute_integer,
    np.floor(
        original_comp[
            COMMUTE
        ].to_numpy()
    ),
)

# -----------------------------
# Structural indicators
# -----------------------------
env = all_comp[
    "Environmental_Concern_Level"
].to_numpy()

home = (
    all_comp[
        "Home_Charging_Possible"
    ]
    == "Yes"
).to_numpy(
    dtype="int8"
)

numeric_features[
    "homecharge"
] = (
    all_comp[
        "Charging_Stations_Near_Home"
    ].to_numpy()
    + 5 * home
).astype("float32")

numeric_features[
    "is_30k_spike"
] = (
    income == 30000
).astype("float32")

numeric_features[
    "is_income_cliff"
] = (
    income >= 170537
).astype("float32")

numeric_features[
    "is_low_income_interval"
] = (
    (income >= 38000)
    & (income <= 42000)
).astype("float32")

numeric_features[
    "is_env_one"
] = (
    env == 1
).astype("float32")


static_features = (
    remove_constant_and_duplicate_columns(
        pd.DataFrame(
            numeric_features
        )
    )
    .astype("float32")
)

print(
    "Static numeric features:",
    static_features.shape[1],
)

## 4. Encoding keys and original-data statistics

The coarse view uses **16 encoding keys**.

Important design choice: raw `Annual_Income_USD` is not itself an encoding key. Income enters the supervised statistics through the four coarse ladder variables.

For every key that exists in the original dataset, the notebook computes:

### Smoothed original target rate

$$
\mathrm{OrigTE}(v)
=
\frac{s_v + 5\mu_{\mathrm{orig}}}
{n_v+5},
$$

where $s_v$ is the number of positive original-data labels for value $v$, $n_v$ is the original-data count, and $\mu_{\mathrm{orig}}$ is the original-data prior.

We also retain the original count. These statistics are independent of competition validation labels.

In [ ]:
key_codes = {}

original_smooth = {}
original_count = {}

original_prior = float(
    original_y.mean()
)

for name, values in raw_keys.items():
    competition_series = pd.Series(
        values
    )

    codes, uniques = pd.factorize(
        competition_series,
        sort=False,
    )

    if len(uniques) <= 1:
        continue

    key_codes[
        name
    ] = codes.astype(
        "int32"
    )

    if name in original_keys:
        stats_frame = pd.DataFrame({
            "key": original_keys[name],
            "y": original_y,
        }).dropna(
            subset=["key"]
        )

        stats = (
            stats_frame
            .groupby(
                "key",
                observed=True,
            )["y"]
            .agg([
                "sum",
                "count",
            ])
        )

        smoothed_rate = (
            stats["sum"]
            + 5 * original_prior
        ) / (
            stats["count"]
            + 5
        )

        original_smooth[name] = (
            competition_series
            .map(smoothed_rate)
            .fillna(original_prior)
            .to_numpy(
                dtype="float32"
            )
        )

        original_count[name] = (
            competition_series
            .map(stats["count"])
            .fillna(0)
            .to_numpy(
                dtype="float32"
            )
        )


keys = remove_constant_and_duplicate_columns(
    pd.DataFrame(
        key_codes
    )
)

# Retain external-stat arrays only for surviving keys.
original_smooth = {
    key: value
    for key, value in original_smooth.items()
    if key in keys.columns
}

original_count = {
    key: value
    for key, value in original_count.items()
    if key in keys.columns
}

print(
    "Encoding keys:",
    keys.shape[1],
)

## 5. Local target-rate windows

A defining part of the coarse view is a second supervised representation based on centered neighborhoods in:

1. annual income;
2. daily commute.

For each query value $x$, target statistics are computed among outer-training observations lying inside an inclusive window

$$
[x-r,\;x+r].
$$

Two versions are created:

- **global** windows;
- windows conditioned on the joint group  
  `City_Type × Current_Car_Type`.

Income radii:

$$
r \in \{2,5,10,25,50,200\}.
$$

Commute radii:

$$
r \in \{1,3,10\}.
$$

For every window, the view produces:

- target rate smoothed by `20`;
- target rate smoothed by `200`;
- `log1p(count)`.

Training-row window features are generated by an **inner 5-fold cross-fit**. Outer validation and competition test features are generated from the estimator fitted on the full outer-training fold.

In [ ]:
class WindowRates:
    INCOME_RADII = (
        2,
        5,
        10,
        25,
        50,
        200,
    )

    COMMUTE_RADII = (
        1,
        3,
        10,
    )

    WINDOWS = (
        INCOME_RADII,
        COMMUTE_RADII,
    )

    @staticmethod
    def _table(
        values,
        targets,
    ):
        order = np.argsort(
            values,
            kind="stable",
        )

        sorted_values = values[
            order
        ]

        prefix_target = np.r_[
            0.0,
            np.cumsum(
                targets[order],
                dtype="float64",
            ),
        ]

        return (
            sorted_values,
            prefix_target,
        )

    def fit(
        self,
        x,
        groups,
        targets,
    ):
        self.prior = float(
            np.mean(targets)
        )

        self.tables = {}

        for col in range(2):
            self.tables[
                col,
                -1,
            ] = self._table(
                x[:, col],
                targets,
            )

            for group in np.unique(
                groups
            ):
                mask = (
                    groups == group
                )

                self.tables[
                    col,
                    int(group),
                ] = self._table(
                    x[mask, col],
                    targets[mask],
                )

        return self

    def transform(
        self,
        x,
        groups,
    ):
        values = []
        names = []

        for col, radii in enumerate(
            self.WINDOWS
        ):
            for conditioned in [
                False,
                True,
            ]:
                group_values = (
                    np.unique(groups)
                    if conditioned
                    else [-1]
                )

                for radius in radii:
                    count = np.zeros(
                        len(x),
                        dtype="float64",
                    )

                    total = np.zeros(
                        len(x),
                        dtype="float64",
                    )

                    for group in group_values:
                        if conditioned:
                            row_idx = np.flatnonzero(
                                groups
                                == group
                            )
                        else:
                            row_idx = np.arange(
                                len(x)
                            )

                        key = (
                            col,
                            int(group),
                        )

                        if key not in self.tables:
                            continue

                        (
                            points,
                            prefix,
                        ) = self.tables[
                            key
                        ]

                        lower = np.searchsorted(
                            points,
                            x[row_idx, col]
                            - radius,
                            side="left",
                        )

                        upper = np.searchsorted(
                            points,
                            x[row_idx, col]
                            + radius,
                            side="right",
                        )

                        count[
                            row_idx
                        ] = (
                            upper
                            - lower
                        )

                        total[
                            row_idx
                        ] = (
                            prefix[upper]
                            - prefix[lower]
                        )

                    prefix_name = (
                        f"window_{col}_"
                        f"{'group' if conditioned else 'global'}_"
                        f"{radius}"
                    )

                    for smooth in [
                        20.0,
                        200.0,
                    ]:
                        rate = (
                            total
                            + smooth
                            * self.prior
                        ) / (
                            count
                            + smooth
                        )

                        values.append(
                            rate.astype(
                                "float32"
                            )
                        )

                        names.append(
                            prefix_name
                            + f"_s{int(smooth)}"
                        )

                    values.append(
                        np.log1p(
                            count
                        ).astype(
                            "float32"
                        )
                    )

                    names.append(
                        prefix_name
                        + "_logcount"
                    )

        return (
            np.column_stack(
                values
            ),
            names,
        )


def crossfit_windows(
    x,
    groups,
    y,
    train_idx,
    valid_idx,
    test_idx,
    inner_folds,
    seed,
):
    inner_cv = StratifiedKFold(
        n_splits=inner_folds,
        shuffle=True,
        random_state=seed,
    )

    train_output = None
    feature_names = None

    # Inner OOF windows for outer-training rows.
    for inner_train_pos, inner_valid_pos in inner_cv.split(
        train_idx,
        y[train_idx],
    ):
        fit_rows = train_idx[
            inner_train_pos
        ]

        score_rows = train_idx[
            inner_valid_pos
        ]

        estimator = WindowRates().fit(
            x[fit_rows],
            groups[fit_rows],
            y[fit_rows],
        )

        encoded, feature_names = (
            estimator.transform(
                x[score_rows],
                groups[score_rows],
            )
        )

        if train_output is None:
            train_output = np.empty(
                (
                    len(train_idx),
                    encoded.shape[1],
                ),
                dtype="float32",
            )

        train_output[
            inner_valid_pos
        ] = encoded

    # Full outer-training fit for validation and test.
    estimator = WindowRates().fit(
        x[train_idx],
        groups[train_idx],
        y[train_idx],
    )

    valid_output = (
        estimator.transform(
            x[valid_idx],
            groups[valid_idx],
        )[0]
    )

    test_output = (
        estimator.transform(
            x[test_idx],
            groups[test_idx],
        )[0]
    )

    return (
        train_output,
        valid_output,
        test_output,
        feature_names,
    )

## 6. Fold-safe Target Encoding and feature assembly

The coarse view uses three smoothing settings:

- `smooth="auto"`
- `smooth=20`
- `smooth=200`

For outer-training rows, `TargetEncoder.fit_transform` performs an inner cross-fit. Outer validation and test rows are transformed by the encoder fitted only on outer-training labels.

The **central** competition Target Encoding is the `smooth=20` version. It is compared with the external smoothed target rate through a log-odds difference:

$$
\mathrm{Lift}(v)
=
\operatorname{logit}
\left(
\mathrm{TE}_{20}^{\mathrm{competition}}(v)
\right)
-
\operatorname{logit}
\left(
\mathrm{OrigTE}(v)
\right).
$$

The external block for each key contains:

- smoothed original target rate;
- `log1p(original_count)`;
- original-novelty indicator `count == 0`;
- competition-vs-original log-odds lift.

The comparison is valid because the competition term is cross-fitted and the original statistics come from an independent external labeled dataset after overlap removal.

In [ ]:
def crossfit_target_encoding(
    key_frame,
    y,
    train_idx,
    valid_idx,
    test_idx,
    smooth_values,
    inner_folds,
    seed,
):
    train_parts = []
    valid_parts = []
    test_parts = []

    feature_names = []
    by_smooth = {}

    for smooth in smooth_values:
        encoder = TargetEncoder(
            target_type="binary",
            smooth=smooth,
            cv=inner_folds,
            shuffle=True,
            random_state=seed,
        )

        train_encoded = encoder.fit_transform(
            key_frame.iloc[
                train_idx
            ],
            y[train_idx],
        ).astype(
            "float32"
        )

        valid_encoded = encoder.transform(
            key_frame.iloc[
                valid_idx
            ]
        ).astype(
            "float32"
        )

        test_encoded = encoder.transform(
            key_frame.iloc[
                test_idx
            ]
        ).astype(
            "float32"
        )

        train_parts.append(
            train_encoded
        )

        valid_parts.append(
            valid_encoded
        )

        test_parts.append(
            test_encoded
        )

        by_smooth[
            str(smooth)
        ] = (
            train_encoded,
            valid_encoded,
            test_encoded,
        )

        feature_names.extend([
            f"TE_{col}_{smooth}"
            for col in key_frame.columns
        ])

    return (
        np.column_stack(
            train_parts
        ),
        np.column_stack(
            valid_parts
        ),
        np.column_stack(
            test_parts
        ),
        feature_names,
        by_smooth,
    )


# Two-column matrix used by the window estimator.
window_x = all_comp[
    [
        INCOME,
        COMMUTE,
    ]
].to_numpy(
    dtype="float64"
)

# Joint group: City_Type × Current_Car_Type.
group_string = (
    all_comp["City_Type"]
    + "|"
    + all_comp[
        "Current_Car_Type"
    ]
)

groups = pd.factorize(
    group_string,
    sort=True,
)[0].astype(
    "int16"
)


def build_fold_matrices(
    train_idx,
    valid_idx,
    seed,
):
    indices = [
        train_idx,
        valid_idx,
        test_idx,
    ]

    parts = [
        [
            static_features.iloc[
                idx
            ].to_numpy(
                dtype="float32"
            )
        ]
        for idx in indices
    ]

    feature_names = list(
        static_features.columns
    )

    # -----------------------------
    # Fold-local, label-free frequencies
    # -----------------------------
    frequency_columns = []

    for col in keys.columns:
        mapping = (
            keys.iloc[
                train_idx
            ][col]
            .value_counts(
                normalize=True
            )
        )

        full_frequency = (
            keys[col]
            .map(mapping)
            .fillna(0)
            .to_numpy(
                dtype="float32"
            )
        )

        frequency_columns.append(
            full_frequency
        )

    frequency_matrix = np.column_stack(
        frequency_columns
    )

    for bucket, idx in zip(
        parts,
        indices,
    ):
        bucket.append(
            frequency_matrix[
                idx
            ]
        )

    feature_names.extend([
        f"freq_{col}"
        for col in keys.columns
    ])

    # -----------------------------
    # Triple cross-fitted TE
    # -----------------------------
    (
        te_train,
        te_valid,
        te_test,
        te_names,
        te_by_smooth,
    ) = crossfit_target_encoding(
        key_frame=keys,
        y=y,
        train_idx=train_idx,
        valid_idx=valid_idx,
        test_idx=test_idx,
        smooth_values=[
            "auto",
            20.0,
            200.0,
        ],
        inner_folds=INNER_FOLDS,
        seed=seed,
    )

    for bucket, encoded in zip(
        parts,
        [
            te_train,
            te_valid,
            te_test,
        ],
    ):
        bucket.append(
            encoded
        )

    feature_names.extend(
        te_names
    )

    # The middle smoothing level is the competition term in the lift.
    central = te_by_smooth[
        "20.0"
    ]

    key_order = list(
        keys.columns
    )

    # -----------------------------
    # External original-data block
    # -----------------------------
    for col in original_smooth:
        key_position = (
            key_order.index(
                col
            )
        )

        for dataset_idx, (
            bucket,
            row_idx,
        ) in enumerate(
            zip(
                parts,
                indices,
            )
        ):
            orig_rate = (
                original_smooth[
                    col
                ][row_idx]
            )

            count = (
                original_count[
                    col
                ][row_idx]
            )

            competition_rate = (
                central[
                    dataset_idx
                ][
                    :,
                    key_position,
                ]
            )

            lift = (
                logit(
                    np.clip(
                        competition_rate,
                        1e-4,
                        1 - 1e-4,
                    )
                )
                - logit(
                    np.clip(
                        orig_rate,
                        1e-4,
                        1 - 1e-4,
                    )
                )
            )

            external_block = np.column_stack([
                orig_rate,
                np.log1p(
                    count
                ),
                count == 0,
                lift,
            ]).astype(
                "float32"
            )

            bucket.append(
                external_block
            )

        feature_names.extend([
            f"original_smooth_{col}",
            f"original_logcount_{col}",
            f"original_novel_{col}",
            f"target_logodds_lift_{col}",
        ])

    # -----------------------------
    # Cross-fitted local windows
    # -----------------------------
    (
        window_train,
        window_valid,
        window_test,
        window_names,
    ) = crossfit_windows(
        x=window_x,
        groups=groups,
        y=y,
        train_idx=train_idx,
        valid_idx=valid_idx,
        test_idx=test_idx,
        inner_folds=INNER_FOLDS,
        seed=seed,
    )

    for bucket, window_matrix in zip(
        parts,
        [
            window_train,
            window_valid,
            window_test,
        ],
    ):
        bucket.append(
            window_matrix
        )

    feature_names.extend(
        window_names
    )

    frames = [
        pd.DataFrame(
            np.column_stack(
                bucket
            ),
            columns=feature_names,
        )
        for bucket in parts
    ]

    # Remove constants according to outer-training rows only.
    keep = [
        col
        for col in frames[0].columns
        if frames[0][col].nunique(
            dropna=False
        ) > 1
    ]

    frames = [
        frame[keep]
        for frame in frames
    ]

    for frame in frames:
        if not np.isfinite(
            frame.to_numpy(
                dtype="float32"
            )
        ).all():
            raise ValueError(
                "Constructed feature matrix contains non-finite values."
            )

    return tuple(
        frames
    )

## 7. LightGBM model

The selected coarse-view LightGBM configuration uses the following regularization / sampling settings:

- `feature_fraction = 0.50`
- `lambda_l2 = 5.0`
- `min_data_in_leaf = 20`

Other important parameters:

- learning rate: `0.02`
- max depth: `5`
- leaves: `32`
- L1: `0.07094`
- max bin: `1024`
- maximum rounds: `20000`
- early stopping patience: `500`

Two seeds are trained: `42` and `152`.

In [ ]:
def fit_lgb_coarse(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    seed,
):
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.02,
        "max_depth": 5,
        "num_leaves": 32,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.50,
        "bagging_freq": 0,
        "lambda_l1": 0.07094,
        "lambda_l2": 5.0,
        "max_bin": 1024,
        "feature_pre_filter": False,
        "verbosity": -1,
        "seed": seed,
        "num_threads": THREADS,
        "deterministic": True,
        "force_col_wise": True,
    }

    dtrain = lgb.Dataset(
        X_train,
        label=y_train,
    )

    dvalid = lgb.Dataset(
        X_valid,
        label=y_valid,
        reference=dtrain,
    )

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=20000,
        valid_sets=[
            dvalid
        ],
        callbacks=[
            lgb.early_stopping(
                500,
                verbose=False,
            ),
            lgb.log_evaluation(
                500
            ),
        ],
    )

    best_iteration = (
        model.best_iteration
        or 20000
    )

    valid_pred = model.predict(
        X_valid,
        num_iteration=best_iteration,
    )

    test_pred = model.predict(
        X_test,
        num_iteration=best_iteration,
    )

    importance = pd.DataFrame({
        "feature": X_train.columns,
        "gain": model.feature_importance(
            importance_type="gain"
        ),
    })

    return (
        valid_pred,
        test_pred,
        best_iteration,
        importance,
    )

## 8. Five-fold OOF training

The outer validation structure is fixed:

$$
\mathrm{StratifiedKFold}
(
5,
\mathrm{shuffle=True},
\mathrm{random\_state}=42
).
$$

For each seed and outer fold, the entire fold-dependent feature pipeline is rebuilt before LightGBM is trained.

In the original run, the final fold matrices contained about **185–186 features**, depending on fold-specific constant removal.

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=FOLD_SEED,
)

folds = list(
    outer_cv.split(
        train,
        y,
    )
)

fold_id = np.empty(
    len(train),
    dtype="int8",
)

for fold, (
    _,
    valid_idx,
) in enumerate(
    folds,
    start=1,
):
    fold_id[
        valid_idx
    ] = fold


seed_oof_predictions = []
seed_test_fold_predictions = []

training_rows = []
importance_frames = []

for seed in MODEL_SEEDS:
    seed_oof = np.zeros(
        len(train),
        dtype="float64",
    )

    seed_test_folds = []

    for fold, (
        train_idx,
        valid_idx,
    ) in enumerate(
        folds,
        start=1,
    ):
        (
            X_train,
            X_valid,
            X_test,
        ) = build_fold_matrices(
            train_idx=train_idx,
            valid_idx=valid_idx,
            seed=seed,
        )

        (
            valid_pred,
            test_pred,
            best_iteration,
            importance,
        ) = fit_lgb_coarse(
            X_train=X_train,
            y_train=y[
                train_idx
            ],
            X_valid=X_valid,
            y_valid=y[
                valid_idx
            ],
            X_test=X_test,
            seed=seed,
        )

        seed_oof[
            valid_idx
        ] = valid_pred

        seed_test_folds.append(
            test_pred
        )

        fold_auc = roc_auc_score(
            y[
                valid_idx
            ],
            valid_pred,
        )

        training_rows.append({
            "seed": seed,
            "fold": fold,
            "auc": fold_auc,
            "best_iteration": best_iteration,
            "n_features": X_train.shape[1],
        })

        importance[
            "seed"
        ] = seed

        importance[
            "fold"
        ] = fold

        importance_frames.append(
            importance
        )

        print(
            f"seed={seed} "
            f"fold={fold} "
            f"AUC={fold_auc:.6f} "
            f"best={best_iteration} "
            f"features={X_train.shape[1]}"
        )

    seed_oof_predictions.append(
        seed_oof
    )

    seed_test_fold_predictions.append(
        np.stack(
            seed_test_folds
        )
    )

## 9. Aggregate seeds and construct rank predictions

The two seeds are averaged at prediction level.

For the later rank ensemble, OOF predictions are ranked independently inside each outer validation fold:

$$
R_i
=
\frac{
\operatorname{rank}(p_i)-0.5
}{
n_{\mathrm{fold}}
}.
$$

The five fold-level test predictions are rank-normalized separately and averaged.

In [ ]:
def normalized_rank(values):
    values = np.asarray(
        values
    )

    return (
        rankdata(
            values,
            method="average",
        )
        - 0.5
    ) / len(values)


# Raw OOF average over seeds.
oof_pred = np.mean(
    seed_oof_predictions,
    axis=0,
)

# seeds × folds × test_rows
test_seed_fold = np.stack(
    seed_test_fold_predictions
)

# Average seeds but retain outer-fold test predictions.
test_fold_pred = np.mean(
    test_seed_fold,
    axis=0,
)

test_pred = np.mean(
    test_fold_pred,
    axis=0,
)

# Fold-normalized OOF ranks.
oof_rank = np.zeros(
    len(train),
    dtype="float64",
)

test_rank = np.zeros(
    len(test),
    dtype="float64",
)

for fold, (
    _,
    valid_idx,
) in enumerate(
    folds
):
    oof_rank[
        valid_idx
    ] = normalized_rank(
        oof_pred[
            valid_idx
        ]
    )

    test_rank += (
        normalized_rank(
            test_fold_pred[
                fold
            ]
        )
        / OUTER_FOLDS
    )


raw_oof_auc = roc_auc_score(
    y,
    oof_pred,
)

rank_oof_auc = roc_auc_score(
    y,
    oof_rank,
)

fold_metrics = pd.DataFrame(
    training_rows
)

print(
    f"Raw OOF AUC:  "
    f"{raw_oof_auc:.6f}"
)

print(
    f"Rank OOF AUC: "
    f"{rank_oof_auc:.6f}"
)

print()
print(
    "Reproduction target:"
)
print(
    "  seed 42 raw OOF  ≈ 0.945761"
)
print(
    "  seed 152 raw OOF ≈ 0.945778"
)
print(
    "  two-seed raw OOF ≈ 0.945831"
)
print(
    "  rank OOF         ≈ 0.945842"
)

## 10. Seed and fold stability

In [ ]:
seed_summary = []

for seed, seed_oof in zip(
    MODEL_SEEDS,
    seed_oof_predictions,
):
    fold_scores = []

    for _, valid_idx in folds:
        fold_scores.append(
            roc_auc_score(
                y[
                    valid_idx
                ],
                seed_oof[
                    valid_idx
                ],
            )
        )

    seed_summary.append({
        "seed": seed,
        "mean_fold_auc": np.mean(
            fold_scores
        ),
        "fold_auc_std": np.std(
            fold_scores
        ),
        "pooled_oof_auc": roc_auc_score(
            y,
            seed_oof,
        ),
    })

seed_summary = pd.DataFrame(
    seed_summary
)

seed_summary

## 11. Feature importance

Gain importance is averaged across all ten trained LightGBM models (`2 seeds × 5 folds`).

This is descriptive only; no feature selection is performed from these validation results.

In [ ]:
importance_all = pd.concat(
    importance_frames,
    ignore_index=True,
)

importance_summary = (
    importance_all
    .groupby(
        "feature",
        as_index=False,
    )["gain"]
    .mean()
    .sort_values(
        "gain",
        ascending=False,
    )
)

importance_summary.head(
    30
)

## 12. Save outputs

These OOF and test files are intended to feed the later ensemble research directly.

In [ ]:
oof_output = pd.DataFrame({
    ID_COL: train[
        ID_COL
    ],
    "y_true": y,
    "fold": fold_id,
    "prediction": oof_pred,
    "prediction_rank": oof_rank,
})

test_output = pd.DataFrame({
    ID_COL: test[
        ID_COL
    ],
    "prediction": test_pred,
    "prediction_rank": test_rank,
})

submission = (
    sample_submission
    .copy()
)

submission[
    TARGET
] = test_pred


oof_output.to_csv(
    OUTPUT_DIR
    / "oof_lgb_coarse.csv",
    index=False,
)

test_output.to_csv(
    OUTPUT_DIR
    / "test_lgb_coarse.csv",
    index=False,
)

fold_metrics.to_csv(
    OUTPUT_DIR
    / "fold_metrics.csv",
    index=False,
)

seed_summary.to_csv(
    OUTPUT_DIR
    / "seed_metrics.csv",
    index=False,
)

importance_summary.to_csv(
    OUTPUT_DIR
    / "feature_importance.csv",
    index=False,
)

submission.to_csv(
    OUTPUT_DIR
    / "submission_lgb_coarse.csv",
    index=False,
)

print(
    "Saved outputs to:",
    OUTPUT_DIR,
)

## 13. Interpretation

`lgb_coarse` is weaker than the strongest individual components, but it represents the problem differently.

Compared with `lgb_exact`, it intentionally:

- removes income digit decomposition;
- removes raw income as a Target-Encoding / frequency key;
- replaces the fine income ladder with `10 / 50 / 500 / 5000`;
- uses fold-local rather than global train+test frequencies;
- uses stronger TE smoothing (`auto / 20 / 200`);
- introduces external original-data support/count/novelty information;
- explicitly measures competition-vs-original target-rate shifts through log-odds lift;
- adds local target-rate windows over income and commute.

That difference in representation is exactly why a model can remain useful to an ensemble even when its standalone AUC is lower.

The component ultimately contributed to the project's ensemble because it supplied a distinct prediction view rather than because it was the strongest standalone model.

## Result

- Raw OOF AUC: **0.945831**
- Rank-normalized OOF AUC: **0.945842**

Although weaker as a standalone model, this component contributes a distinct prediction view to the final ensemble analysis in
`06_final_research.ipynb`.